# Running an input file

The quickest way to run pyRADDOSE-3D is the way RADDOSE-3D is run: hand it
an input file, then read the summary it prints and the files it writes.
`inputs/`, beside this notebook, holds four, ready to run:

| file | experiment |
|---|---|
| `insulin.txt` | RADDOSE-3D's own example input, `MyInput.txt`, unchanged: an insulin crystal in an X-ray beam, two wedges |
| `insulin.yaml` | the same experiment as a YAML configuration |
| `microed.txt` | RADDOSE-3D's `ED_input_cell.txt`, unchanged: a protein microcrystal in a 300 keV electron beam |
| `microed.yaml` | the same, as YAML |

Any RADDOSE-3D input file is read the same way.

## An input file

This is `insulin.txt`:

In [ ]:
from pathlib import Path

INPUTS = Path("inputs")
print((INPUTS / "insulin.txt").read_text())

Read it, run it, and print what RADDOSE-3D prints:

In [ ]:
from pyraddose_3d.models.experiment import Experiment
from pyraddose_3d.outputs import summary_text
from pyraddose_3d.solve import simulate

experiment = Experiment.from_file(INPUTS / "insulin.txt")
simulation = simulate(experiment, record_images=True)
print(summary_text(simulation))

`write_outputs` writes RADDOSE-3D's files, under its names and in its
formats. Here they go to a temporary directory. `prefix` works as
RADDOSE-3D's `--prefix` does, so `prefix="output-"` would write
`output-Summary.txt` and the rest to the current directory.

In [ ]:
import tempfile

from pyraddose_3d.outputs import write_outputs

directory = Path(tempfile.mkdtemp())
for path in write_outputs(simulation, prefix=str(directory / "output-")):
    print(path.name)

## The same experiment in YAML

A YAML configuration says the same thing with each setting named, and its
units and RADDOSE-3D keyword in a comment. `from_file` reads anything
ending in `.yaml` or `.yml` as one, and anything else as a RADDOSE-3D input
file:

In [ ]:
print((INPUTS / "insulin.yaml").read_text())

In [ ]:
Experiment.from_file(INPUTS / "insulin.yaml") == experiment

## MicroED

`Subprogram EMED` in an input file (`subprogram: microed` in YAML) asks for
RADDOSE-3D's MicroED engine, which `microed` runs:

In [ ]:
print((INPUTS / "microed.txt").read_text())

In [ ]:
from pyraddose_3d.outputs import microed_report
from pyraddose_3d.solve import microed

print(microed_report(microed(Experiment.from_file(INPUTS / "microed.txt"))))

## From the command line

`programs/dose/run_dose.py` does all of this for any input -- MX, MicroED,
or the Monte Carlo and XFEL subprograms -- printing the summary and writing
the files to the current directory. From the top of the repository:

```sh
python programs/dose/run_dose.py docs/examples/inputs/insulin.txt
```

With no input named, it runs `insulin.txt`. Its other settings, the output
prefix and whether to write the per-image files, are at the top of the
script.